In [ ]:

import json
import pandas as pd

with open("../logs/results_summary.json") as f:
    data = json.load(f)

df = pd.DataFrame(data)
df[["label", "lsb_payload", "blocked", "response"]]

In [ ]:
with open("../logs/scanner_report.json") as f:
    scan_data = json.load(f)

scan_df = pd.DataFrame(scan_data)

# Flatten all nested fields at once
scan_df["lsb_detected"]             = scan_df["lsb"].apply(lambda x: x["detected"])
scan_df["exif_detected"]            = scan_df["exif"].apply(lambda x: x["detected"])
scan_df["entropy_ratio"]            = scan_df["entropy"].apply(lambda x: x["lsb_ones_ratio"])
scan_df["channel_std"]              = scan_df["entropy"].apply(lambda x: x.get("channel_std", 0))
scan_df["entropy_flagged"]          = scan_df["entropy"].apply(lambda x: x["entropy_flag"])

scan_df[[
    "file","lsb_detected","exif_detected",
    "entropy_ratio","channel_std","entropy_flagged",
    "sanitized_lsb_survives","sanitized_exif_survives"
]]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

files = scan_df["file"]
x     = np.arange(len(files))
w     = 0.25

fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(x - w,  scan_df["lsb_detected"].astype(int),             w, label="LSB detected",              color="#e05c5c")
ax.bar(x,      scan_df["exif_detected"].astype(int),            w, label="EXIF detected",             color="#e09a5c")
ax.bar(x + w,  scan_df["sanitized_lsb_survives"].astype(int),   w, label="LSB survives sanitization", color="#5c7ae0")

ax.set_xticks(x)
ax.set_xticklabels(files, rotation=35, ha="right")
ax.set_yticks([0, 1])
ax.set_yticklabels(["No", "Yes"])
ax.set_title("Detection robustness across embedding methods")
ax.legend()
plt.tight_layout()
plt.savefig("../logs/detection_chart.png", dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ["#e05c5c" if f else "#5cb85c" for f in scan_df["entropy_flagged"]]

# Left panel — overall ratio
axes[0].bar(scan_df["file"], scan_df["entropy_ratio"], color=colors)
axes[0].axhline(0.50, color="black",  linestyle="--", linewidth=1, label="Stego target (0.5)")
axes[0].axhline(0.48, color="orange", linestyle=":",               label="Threshold band")
axes[0].axhline(0.52, color="orange", linestyle=":")
axes[0].set_ylabel("LSB ones ratio")
axes[0].set_title("Overall LSB entropy\nRed = flagged suspicious")
axes[0].legend(fontsize=8)
axes[0].tick_params(axis='x', rotation=45)

# Right panel — per-channel std
axes[1].bar(scan_df["file"], scan_df["channel_std"], color=colors)
axes[1].axhline(0.01, color="red", linestyle="--", label="Suspicion threshold (0.01)")
axes[1].set_ylabel("Channel std deviation")
axes[1].set_title("Per-channel LSB variance\nLow variance = stego-like")
axes[1].legend(fontsize=8)
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig("../logs/entropy_chart.png", dpi=150)
plt.show()

In [ ]:
summary = scan_df[[
    "file",
    "lsb_detected",
    "exif_detected",
    "entropy_flagged",
    "sanitized_lsb_survives",
    "sanitized_exif_survives",
    "deobfuscated"
]].copy()

summary.columns = [
    "Image",
    "LSB Found",
    "EXIF Found",
    "Entropy Flagged",
    "LSB Survives Sanitize",
    "EXIF Survives Sanitize",
    "Obfuscation Decoded"
]

summary

In [ ]:
print("=" * 55)
print("  RESEARCH FINDINGS SUMMARY")
print("=" * 55)

lsb_detected  = scan_df["lsb_detected"].sum()
exif_detected = scan_df["exif_detected"].sum()
lsb_survives  = scan_df["sanitized_lsb_survives"].sum()
exif_survives = scan_df["sanitized_exif_survives"].sum()
total         = len(scan_df)

print(f"\n  Total images scanned   : {total}")
print(f"  LSB payloads detected  : {lsb_detected}/{total}")
print(f"  EXIF payloads detected : {exif_detected}/{total}")
print(f"\n  After PIL sanitization:")
print(f"  LSB payloads survive   : {lsb_survives}/{total}")
print(f"  EXIF payloads survive  : {exif_survives}/{total}")

print(f"\n  FINDINGS:")

if lsb_detected > 0:
    print(f"  [+] LSB steganography detectable via entropy analysis")
if lsb_survives > 0:
    print(f"  [!] LSB payloads SURVIVE PIL sanitization — CRITICAL GAP")
else:
    print(f"  [+] LSB payloads removed by sanitization")

if exif_detected > 0:
    print(f"  [+] EXIF payloads detected successfully")
    if exif_survives == 0:
        print(f"  [+] EXIF payloads removed by sanitization — effective defense")
    else:
        print(f"  [!] EXIF payloads survive sanitization")
else:
    print(f"  [-] EXIF detection failed — check encoder (PNG vs JPEG issue)")

print(f"\n  CONCLUSION:")
if lsb_survives > 0 and exif_detected > 0 and exif_survives == 0:
    print(f"  Asymmetric defense gap confirmed:")
    print(f"  EXIF: detectable + removable via sanitization")
    print(f"  LSB:  detectable but NOT removable via sanitization")
    print(f"  LSB requires pixel-level noise injection as defense.")
elif lsb_survives > 0:
    print(f"  LSB payloads survive sanitization.")
    print(f"  EXIF results inconclusive — verify encoder.")
else:
    print(f"  Results inconclusive — re-run after fixing encoder.")

print("=" * 55)